# NB23 — Commercial cleanup investigation (Methodist / Parkland)

**Purpose.** A *read-only diagnostic* that surfaces, characterizes, and validates the three known
contamination patterns in the assembled commercial **73721** (MRI lower-extremity, no contrast)
rate rows that feed **NB16 → NB17 → NB18** (the insurance-negotiation / commercial-spread driver).

NB23 **recomputes nothing** and **mutates no parent**. It reads NB16's assembled commercial rows,
applies three explicit cleanup *rules* as testable predicates, and emits a cleanup **manifest**
(`outputs/nb23_cleanup.json`) plus a *proposed* clean set for inspection. The canonical cleanup is
then ported **upstream into NB16** — same "fix upstream, not in the derived notebook" principle used
for the NB22 crosswalk.

### The three rules under investigation
1. **R1 — Methodist 232.47 MA drop.** Methodist (CCN 450051) carries six rows priced at `$232.47`,
   a near-Medicare amount attached to Medicare-Advantage-style payers that are contaminating the
   commercial book. Drop them. **Expected: 6.**
2. **R2 — AETNA transplant carve-out.** AETNA rows tagged to a transplant service line / network are
   not standard commercial 73721 negotiated rates. Carve them out. **Expected: discovered live.**
3. **R3 — Parkland %-of-charges exclusion.** Parkland (CCN 450015) publishes some "rates" as a
   *percent of billed charges* (`methodology_normalized`), not comparable dollar negotiated rates.
   Exclude them from the dollar-based commercial spread. **Expected: discovered live.**

### Gated state (inherited)
NB23 is only as trustworthy as the raw rows it reads. It inherits NB16's gated state and publishes a
**do-not-publish** block unless `CLEANUP_VERIFIED` passes every tripwire (chiefly: Methodist drop == 6,
zero clean-row collateral, and the source is actually present).

> **Standard mold.** Step 0 (paths) · Step 1 (adaptive recon) · Step 2 (rule contract) ·
> Step 3 (synthetic smoke) · Step R (real apply + gate) · Step S (toggles) · Step V (before/after) ·
> Step F (manifest payload) · Step I (integrity).

## Step 0 — ROOT resolution, paths, source discovery

No computation, no mutation. Resolve `ROOT` (repo `src/queries.py` marker if present, else the
notebook directory), derive `OUTPUTS`, discover NB16's assembled commercial-rows source (tolerant to
JSON or CSV), and set the `NB23_JSON` target. `REAL_AVAILABLE` reflects whether a real source was found;
when it is not, the notebook still runs its synthetic smoke and reports a gated result.

In [ ]:
# ── NB23 · Step 0 — Source loader: five DuckDBs → stamped `raw_rows` ──────────────
# Read-only against data/raw/*_parsed.duckdb. Loops query_procedure_rates_agg + add_lob,
# stamps ccn/hospital/code (NOT in the query output), concats → `raw_rows`, writes
# outputs/nb23_rows.json for Step 1. Mutates no parent.
import sys, glob, importlib
from pathlib import Path
import pandas as pd
import duckdb

# resolve repo root (folder containing data/raw) + outputs
here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "data" / "raw").is_dir()), here)
OUTPUTS = ROOT / "outputs"; OUTPUTS.mkdir(exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print(f"ROOT={ROOT}")

TARGET_CODE = "73721"

# ccn ↔ dbfile registry (filename-stem substring -> ccn, hospital label). STILL UNVERIFIED.
REGISTRY = {
    "baylor_university_medical_center":         ("450021", "Baylor University Medical Center"),
    "medical_city_alliance_hospital":           ("670103", "Medical City Alliance"),
    "methodist_dallas_medical_center":          ("450051", "Methodist Dallas Medical Center"),
    "parkland_health":                          ("450015", "Parkland Health"),
    "texas_health_presbyterian_hospital_plano": ("450771", "Texas Health Presbyterian Plano"),
}

def _imp(name, modules):
    for m in modules:
        try:
            mod = importlib.import_module(m)
            if hasattr(mod, name): return getattr(mod, name)
        except Exception: pass
    return None

query_agg = _imp("query_procedure_rates_agg", ["src.queries", "queries"])
add_lob   = (_imp("add_lob",    ["src.queries", "queries", "src.lob", "src.payers", "src.classify"])
          or _imp("add_lob_v4", ["src.queries", "queries", "src.lob", "src.payers", "src.classify"]))
if query_agg is None:
    raise ImportError("query_procedure_rates_agg not found — check the module path under src/.")
if add_lob is None:
    print("WARN: add_lob not found — proceeding without lob (payer_class falls back to heuristic).")

def _reg_for(stem):
    for key, val in REGISTRY.items():
        if key in stem: return val
    return (None, None)

def _apply_lob(df, con):
    if add_lob is None: return df
    for call in (lambda: add_lob(df), lambda: add_lob(con, df)):
        try:
            res = call()
            return res if isinstance(res, pd.DataFrame) else df
        except TypeError:
            continue
        except Exception as e:
            print(f"    add_lob failed: {e}"); return df
    return df

def _query(con):
    for kw in ({"setting": "outpatient"}, {}):
        try:
            return query_agg(con, TARGET_CODE, **kw)
        except TypeError:
            continue
    return query_agg(con, TARGET_CODE)

frames = []
for path in sorted(glob.glob(str(ROOT / "data" / "raw" / "*_parsed.duckdb"))):
    stem = Path(path).name.replace("_parsed.duckdb", "")
    ccn, hosp = _reg_for(stem)
    con = duckdb.connect(path, read_only=True)
    try:
        df = _query(con)
        df = _apply_lob(df, con)
    finally:
        con.close()
    if df is None or len(df) == 0:
        print(f"  {stem:<44} 0 rows"); continue
    df = df.copy()
    df["ccn"], df["hospital"], df["code"] = ccn, hosp, TARGET_CODE
    frames.append(df)
    print(f"  {stem:<44} {len(df):>4} rows  -> ccn {ccn}")

raw_rows = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
raw_rows.to_json(OUTPUTS / "nb23_rows.json", orient="records")
print(f"\nraw_rows: {len(raw_rows)} rows, {raw_rows.shape[1]} cols")
print(f"cols: {list(raw_rows.columns)}")
print(f"wrote {OUTPUTS / 'nb23_rows.json'}")

In [ ]:
# --- scratch: locate NB16's commercial-rows artifact -------------------------
print("OUTPUTS contents:")
for p in sorted(OUTPUTS.glob("*")):
    print(f"   {p.name:<40} {p.stat().st_size:>10} bytes")

print("\nDATA contents (if present):")
if DATA.exists():
    for p in sorted(DATA.glob("*")):
        print(f"   {p.name:<40} {p.stat().st_size:>10} bytes")
else:
    print("   (no data/ dir)")

# peek at any nb16-ish json so we can see if it holds row-level rates
import json as _json
for p in sorted(OUTPUTS.glob("*nb16*")) + sorted(OUTPUTS.glob("*commercial*")) + sorted(OUTPUTS.glob("*16*")):
    try:
        obj = _json.loads(p.read_text())
    except Exception as e:
        print(f"\n{p.name}: not json ({e})"); continue
    print(f"\n=== {p.name} ===  top-type={type(obj).__name__}")
    if isinstance(obj, dict):
        print("   keys:", list(obj.keys())[:20])
        for k, v in obj.items():
            if isinstance(v, list) and v and isinstance(v[0], dict):
                print(f"   -> list '{k}' has {len(v)} records; first record keys:", list(v[0].keys()))
    elif isinstance(obj, list) and obj:
        print(f"   list of {len(obj)} records; first record keys:", list(obj[0].keys()) if isinstance(obj[0], dict) else type(obj[0]).__name__)

In [ ]:
# --- scratch 2: find the raw commercial rows + read the diagnostics ----------
print("=== data/ recursive ===")
for p in sorted(DATA.rglob("*")):
    tag = "DIR " if p.is_dir() else f"{p.stat().st_size:>10}B"
    print(f"   {tag}  {p.relative_to(DATA)}")

print("\n=== payer_columns_diagnostic.txt (first 60 lines) ===")
diag = OUTPUTS / "payer_columns_diagnostic.txt"
if diag.exists():
    print("\n".join(diag.read_text(errors='replace').splitlines()[:60]))

print("\n=== unknown_payers_v3.txt (full) ===")
up = OUTPUTS / "unknown_payers_v3.txt"
if up.exists():
    print(up.read_text(errors='replace')[:1500])

# how does NB16 load its rows? scan src for loaders / file paths
print("\n=== src/queries.py load hints ===")
qp = SRC / "queries.py"
if qp.exists():
    for i, ln in enumerate(qp.read_text(errors='replace').splitlines()):
        if re.search(r"def |read_csv|read_parquet|\.json|/raw|/processed|73721|glob|open\(", ln):
            print(f"   {i:>3}: {ln.strip()[:110]}")

In [ ]:
# --- scratch 3: inspect queries.py API + DuckDB schema -----------------------
import duckdb
qsrc = (SRC / "queries.py").read_text(errors="replace")
lines = qsrc.splitlines()

print("=== module-level defs & constants ===")
for i, ln in enumerate(lines):
    if re.match(r"^(def |[A-Z_][A-Z0-9_]*\s*=|HOSP|CCN|DB_|RAW|PATH)", ln):
        print(f"  {i:>3}: {ln.strip()[:120]}")

def _body(name):
    out, cap = [], False
    for ln in lines:
        if re.match(rf"^def {name}\b", ln): cap = True
        elif cap and re.match(r"^\S", ln) and not ln.startswith(("def "+name,)) and ln[0] not in " \t#":
            if re.match(r"^def ", ln): break
        if cap: out.append(ln)
        if cap and re.match(r"^def ", ln) and not re.match(rf"^def {name}\b", ln): break
    return "\n".join(out[:70])

for fn in ("query_procedure_rates_agg", "add_lob_v4", "add_lob", "classify_plan"):
    print(f"\n=== def {fn} ===")
    print(_body(fn))

# --- DuckDB schema: open Methodist read-only, list tables + columns ----------
mfile = next((DATA/"raw").glob("methodist*parsed.duckdb"), None)
print(f"\n=== Methodist DuckDB = {mfile.name if mfile else None} ===")
if mfile:
    con = duckdb.connect(str(mfile), read_only=True)
    tabs = con.execute("SHOW TABLES").fetchall()
    print("tables:", [t[0] for t in tabs])
    for t in tabs:
        cols = con.execute(f"PRAGMA table_info('{t[0]}')").fetchall()
        print(f"  {t[0]}: {[c[1] for c in cols]}")
    con.close()

In [ ]:
# --- scratch 4: pull 73721 from Methodist via queries.py, inspect the table --
import duckdb
import pandas as pd
pd.set_option("display.max_columns", None); pd.set_option("display.width", 240)

raw = DATA / "raw"
mfile = next(raw.glob("methodist*parsed.duckdb"))
con = duckdb.connect(str(mfile), read_only=True)

df = _q.query_procedure_rates_agg(con, "73721")
print("=== query_procedure_rates_agg('73721') ===")
print("shape:", df.shape)
print("columns:")
for c in df.columns:
    print(f"   {c:<28} {str(df[c].dtype):<10} e.g. {repr(df[c].dropna().iloc[0]) if df[c].notna().any() else 'NaN'}")

# apply the current LOB classifier (prefer the unversioned alias, fall back to v4)
dfl, lobsrc = None, None
for fn in ("add_lob", "add_lob_v4", "add_lob_v3"):
    if hasattr(_q, fn):
        try:
            dfl = getattr(_q, fn)(df.copy()); lobsrc = fn; break
        except Exception as e:
            print(f"   {fn} failed: {e}")
print(f"\nLOB via {lobsrc} -> added cols:", [c for c in dfl.columns if c not in df.columns] if dfl is not None else None)

# locate the rate column and look for the 232.47 cluster
ratecol = next((c for c in df.columns if re.search(r"(neg.*rate|rate|dollar|amount|charge|median|price)", c, re.I)), None)
print("likely rate column:", ratecol)
if ratecol and dfl is not None:
    vals = pd.to_numeric(dfl[ratecol], errors="coerce").round(2)
    hit = dfl[vals == 232.47]
    print(f"\nrows at 232.47 = {len(hit)}")
    show = [c for c in ("payer_name","payer","payer_group","payer_type","plan_name","lob",ratecol,"methodology","setting") if c in dfl.columns]
    print(hit[show].to_string(index=False))
    print("\nLOB distribution:", dfl["lob"].value_counts().to_dict() if "lob" in dfl.columns else "no lob col")
con.close()

In [ ]:
# --- scratch 5: prototype Step-1 loader + ground-truth the 3 patterns --------
import duckdb, pandas as pd, numpy as np
pd.set_option("display.max_columns", None); pd.set_option("display.width", 260)

raw = DATA / "raw"
DB_CCN = {   # dbfile prefix -> CCN  (crosswalk still to be verified upstream)
    "450021": "baylor_university_medical_center",
    "670103": "medical_city_alliance_hospital",
    "450051": "methodist_dallas_medical_center",
    "450015": "parkland_health",
    "450771": "texas_health_presbyterian_hospital_plano",
}
def load_ccn(ccn):
    f = next(raw.glob(f"{DB_CCN[ccn]}*parsed.duckdb"))
    con = duckdb.connect(str(f), read_only=True)
    df = _q.add_lob(_q.query_procedure_rates_agg(con, "73721"))
    con.close()
    df.insert(0, "ccn", ccn)
    return df

allrows = pd.concat([load_ccn(c) for c in DB_CCN], ignore_index=True)
print("TOTAL rows:", len(allrows), "| per ccn:", allrows["ccn"].value_counts().to_dict())
print("lob overall:", allrows["lob"].value_counts().to_dict())

cols = ["ccn","payer_name","payer_group","payer_type","plan_name","lob","dollar_rate","pct_rate","methodology_normalized"]

# 1) Methodist: the 232.47 cluster (against dollar_rate) + all its dollar rows
m = allrows[allrows.ccn == "450051"].copy()
n2347 = int((m.dollar_rate.round(2) == 232.47).sum())
print(f"\n== Methodist dollar_rate==232.47 : {n2347} rows")
print(m[m.dollar_rate.round(2) == 232.47][cols].to_string(index=False))
print("\n== Methodist ALL dollar-rate rows (sorted), to see the cluster in context:")
print(m[m.dollar_rate.notna()][cols].sort_values("dollar_rate").to_string(index=False))

# 2) Parkland: percent-of-charges rows
p = allrows[allrows.ccn == "450015"].copy()
print("\n== Parkland pct_rate set / dollar_rate null:")
print(p[p.dollar_rate.isna() & p.pct_rate.notna()][cols].to_string(index=False))
print("Parkland methodology_normalized counts:", p.methodology_normalized.value_counts().to_dict())

# 3) AETNA transplant scan across all hospitals
desc = allrows["descriptions"] if "descriptions" in allrows.columns else pd.Series([""]*len(allrows))
blob = (allrows.payer_name.fillna("") + " " + allrows.plan_name.fillna("") + " " + desc.fillna("")).str.upper()
aet = allrows[allrows.payer_name.str.upper().str.contains("AETNA", na=False) & blob.str.contains("TRANSPLANT")]
print(f"\n== AETNA transplant rows: {len(aet)}")
print(aet[cols].to_string(index=False))

## Step 1 — Adaptive recon: tolerant field discovery → normalized long table

Load rows (JSON list-of-records or CSV), then discover the columns we need by candidate-name matching
so the join survives cross-parent schema drift (Day-29 lesson: *derive keys from the artifacts, don't
re-eyeball them*). Build a normalized long table with canonical columns: `ccn`, `hospital`, `payer_norm`,
`plan`, `code`, `rate_value` (dollars), `rate_pct` (percent-of-charges), `methodology_norm`,
`payer_class` (heuristic), and a searchable `text_blob`. CCN is the join key; hospital names are labels.

The CCN→name mapping is **not ground-truthed here** (deferred upstream): we assert only the three CCNs
the rules target — Methodist `450051`, Parkland `450015`, MCA `670103` — and read the rest from data.

In [ ]:
# --- resolve OUTPUTS/ROOT if Step 0 hasn't run in this kernel ---
if "OUTPUTS" not in globals():
    if "ROOT" in globals():
        ROOT = Path(ROOT)
    else:
        here = Path.cwd()
        ROOT = next((p for p in (here, *here.parents) if (p / "data" / "raw").is_dir()), here)
    OUTPUTS = ROOT / "outputs"
    OUTPUTS.mkdir(exist_ok=True)
    print(f"resolved ROOT={ROOT}  OUTPUTS={OUTPUTS}")

In [ ]:
# ── NB23 · Step 1 — Adaptive recon: tolerant field discovery → normalized long table ──
import json, re
from pathlib import Path
import pandas as pd

# Source: a JSON list-of-records or CSV that Step 0 wrote (in-memory `raw_rows` also tolerated).
SRC_CANDIDATES = [OUTPUTS / "nb23_rows.json", OUTPUTS / "nb23_rows.csv"]
TARGET_CODE = "73721"
RULE_CCNS = {"450051": "Methodist Dallas", "450015": "Parkland", "670103": "MCA"}  # asserted present

def load_records():
    for p in SRC_CANDIDATES:
        if Path(p).exists():
            if Path(p).suffix == ".json":
                data = json.loads(Path(p).read_text())
                if isinstance(data, dict):
                    for k in ("rows", "data", "records"):
                        if isinstance(data.get(k), list): data = data[k]; break
                return list(data), str(p)
            return pd.read_csv(p).to_dict("records"), str(p)
    obj = globals().get("raw_rows")
    if isinstance(obj, pd.DataFrame): return obj.to_dict("records"), "raw_rows (DataFrame)"
    if isinstance(obj, list):         return obj, "raw_rows (list)"
    raise FileNotFoundError(f"No rows source in {[str(p) for p in SRC_CANDIDATES]} or `raw_rows`.")

records, src = load_records()

# Tolerant field discovery: canonical -> ordered candidate source names (schema-drift proof).
CANDIDATES = {
    "ccn":              ["ccn","cms_certification_number","provider_ccn","provider_id","provider_number"],
    "hospital":         ["hospital","hospital_name","facility_name","provider_name","source","source_file"],
    "payer_norm":       ["payer_norm","payer_name","payer","payer_group"],
    "plan":             ["plan","plan_name"],
    "code":             ["code","billing_code","cpt","hcpcs","procedure_code"],
    "rate_value":       ["rate_value","dollar_rate","negotiated_rate","standard_charge_negotiated_dollar","rate"],
    "rate_pct":         ["rate_pct","pct_rate","percent_of_charges","standard_charge_negotiated_percent"],
    "methodology_norm": ["methodology_norm","methodology_normalized","methodology","standard_charge_methodology"],
    "payer_class":      ["lob","payer_class","line_of_business","payer_type"],  # reuse add_lob output if present
    "descriptions":     ["descriptions","description","code_description"],
}
nk = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
idx = {nk(k): k for k in records[0].keys()}
colmap = {c: next((idx[nk(x)] for x in cands if nk(x) in idx), None) for c, cands in CANDIDATES.items()}

NAME2CCN = [(re.compile(p, re.I), c) for c, p in {
    "450021": r"baylor", "670103": r"medical\s*city|alliance|\bmca\b", "450051": r"methodist",
    "450015": r"parkland", "450771": r"presbyterian|plano|texas\s*health|\bthr\b"}.items()]

def num(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return None
    if isinstance(v, (int, float)): return float(v)
    s = re.sub(r"[,$%\s]", "", str(v))
    try: return float(s) if s.lower() not in ("", "nan", "none", "null") else None
    except ValueError: return None

def g(rec, c): return rec.get(colmap[c]) if colmap[c] else None

def ccn_of(rec):
    raw = g(rec, "ccn")
    if raw is not None:
        m = re.search(r"\d{5,6}", str(raw))
        if m: return m.group(0)
    lab = g(rec, "hospital")
    if lab:
        for rx, c in NAME2CCN:
            if rx.search(str(lab)): return c
    return None

def payer_class(rec, blob):
    v = g(rec, "payer_class")
    if v and str(v).strip(): return str(v)            # prefer repo classifier (lob/payer_type)
    t = blob.lower()
    if re.search(r"medicaid|\bchip\b|\bstar\b", t): return "medicaid"
    if re.search(r"advantage|medicare\s*adv|part\s*[cd]|\bma\b", t): return "medicare_advantage"
    if re.search(r"medicare", t): return "medicare"
    if re.search(r"exchange|marketplace|\baca\b", t): return "aca"
    if re.search(r"ppo|hmo|epo|commercial|bcbs|aetna|cigna|united|humana", t): return "commercial"
    return "unknown"

norm = []
for rec in records:
    payer, plan = g(rec, "payer_norm"), g(rec, "plan")
    meth, desc  = g(rec, "methodology_norm"), g(rec, "descriptions")
    blob = " | ".join(str(x) for x in (payer, plan, meth, desc) if x is not None)
    norm.append({
        "ccn":              ccn_of(rec),
        "hospital":         payer if False else g(rec, "hospital"),
        "payer_norm":       payer,
        "plan":             plan,
        "code":             (str(g(rec, "code")).strip() if g(rec, "code") is not None else None),
        "rate_value":       num(g(rec, "rate_value")),
        "rate_pct":         num(g(rec, "rate_pct")),
        "methodology_norm": (str(meth).strip().lower() if meth is not None else None),
        "payer_class":      payer_class(rec, blob),
        "text_blob":        blob,
    })
long_df = pd.DataFrame(norm)

# Tolerant code narrow to the procedure under investigation.
if colmap["code"] and long_df["code"].nunique(dropna=True) > 1 and (long_df["code"] == TARGET_CODE).any():
    long_df = long_df[long_df["code"] == TARGET_CODE].reset_index(drop=True)

# Recon + assert the three rule-target CCNs (rest read from data).
print(f"source: {src}  |  rows: {len(long_df)}")
print("colmap:", colmap)
print("\nrows per CCN:\n", long_df["ccn"].value_counts(dropna=False).to_string())
print("\nmethodology_norm mix:\n", long_df["methodology_norm"].value_counts(dropna=False).to_string())
present = set(long_df["ccn"].dropna().astype(str))
missing = [c for c in RULE_CCNS if c not in present]
assert not missing, f"rule-target CCNs missing: {[(c, RULE_CCNS[c]) for c in missing]}"
print("\nOK — rule-target CCNs present:", sorted(present & set(RULE_CCNS)))

In [ ]:
# ── NB23 · Placeholder-floor diagnostic (feeds R1; ratifies D84 per-hospital scope) ──
# Per hospital: the dominant fee-schedule $ stamped across many payers = the placeholder
# floor R1 drops. Read-only on long_df. Builds PLACEHOLDER_FLOORS for Step 2.
FEE = "fee schedule"
MIN_ROWS, MIN_PAYERS = 3, 3          # a floor must repeat >=MIN_ROWS across >=MIN_PAYERS payers

fee = long_df[long_df["methodology_norm"] == FEE].copy()
fee["rate_r"] = fee["rate_value"].round(2)

PLACEHOLDER_FLOORS = {}               # ccn -> {value, count, payers}
print(f"{'ccn':<8}{'hospital':<32}{'fee_rows':>9}{'floor$':>11}{'n':>5}{'payers':>8}{'%fee':>7}")
for ccn, g in fee.groupby("ccn"):
    gd = g.dropna(subset=["rate_r"])
    hosp = str(g["hospital"].iloc[0])[:31]
    if gd.empty:
        print(f"{ccn:<8}{hosp:<32}{len(g):>9}{'(no $)':>11}"); continue
    stats = (gd.groupby("rate_r")
               .agg(n=("rate_r", "size"), payers=("payer_norm", "nunique"))
               .sort_values("n", ascending=False))
    val, top = float(stats.index[0]), stats.iloc[0]
    print(f"{ccn:<8}{hosp:<32}{len(g):>9}{val:>11.2f}{int(top['n']):>5}{int(top['payers']):>8}{100*top['n']/len(g):>6.0f}%")
    if top["n"] >= MIN_ROWS and top["payers"] >= MIN_PAYERS:
        PLACEHOLDER_FLOORS[ccn] = {"value": val, "count": int(top["n"]), "payers": int(top["payers"])}
    # show top-3 fee-schedule values so dominance is eyeball-checkable
    for rv, r in stats.head(3).iterrows():
        print(f"          {rv:>11.2f}  n={int(r['n']):<3} payers={int(r['payers'])}")

print("\nPLACEHOLDER_FLOORS (R1 drop targets):")
for ccn, d in PLACEHOLDER_FLOORS.items():
    print(f"  {ccn}: ${d['value']:.2f}  ({d['count']} rows across {d['payers']} payers)")

# reconcile the Day-29 'six 232.47 MA rows' note
m232 = long_df[(long_df["ccn"] == "450051") & (long_df["rate_value"].round(2) == 232.47)]
print(f"\nMethodist 232.47: {len(m232)} rows, {m232['payer_norm'].nunique()} distinct payers "
      f"(Day-29 note='six MA rows'; Day-30='58 placeholder rows').")

## Step 2 — Contract: the three cleanup rules as testable predicates

Each rule is a pure function `long → boolean mask` plus metadata (`id`, `description`, `target_ccn`,
`expected` count, `params`). `apply_cleanup` runs all three, records every dropped row with its
provenance and reason into a manifest, and returns the kept set. Rules never mutate `long`.

Guards: a **planted tripwire** (a known-clean commercial dollar row must survive); the raw procedure
code (`73721`) is only ever a *column value / caveat*, never a manifest dict **key** (D44/D71 analog);
`assess`-style dollar comparisons ignore rows whose `rate_value` is NaN (percent-of-charges rows).
Self-checks run all three rules on a tiny inline example and assert the expected catches.

In [ ]:
# ── NB23 · Step 2 — Cleanup rules as testable predicates (R1/R2/R3) → clean_df ────
# R1 placeholder floor  — dominance-gated → Methodist only (D84 + Day-31 decision)
# R2 AETNA transplant   — specialty carve-out; kept separate for provenance (overlaps R3)
# R3 percent-of-charges — not dollar-comparable; dataset-wide
# Read-only on long_df. Mutates no parent.
FEE, PCT = "fee schedule", "percent of total billed charges"
DOMINANCE_PCT = 0.50           # a fee-schedule value is a placeholder only if it dominates its hospital

# -- R1: dominant placeholder floor per hospital (Methodist only under this gate) --
DOMINANT_FLOORS = {}           # ccn -> floor $
_fee = long_df[long_df["methodology_norm"] == FEE].dropna(subset=["rate_value"]).copy()
_fee["rate_r"] = _fee["rate_value"].round(2)
for ccn, g in _fee.groupby("ccn"):
    vc = g["rate_r"].value_counts()
    if vc.iloc[0] / len(g) >= DOMINANCE_PCT:
        DOMINANT_FLOORS[ccn] = float(vc.index[0])
print("R1 dominant placeholder floors:", DOMINANT_FLOORS)

def r1_mask(df):
    m = pd.Series(False, index=df.index)
    for ccn, val in DOMINANT_FLOORS.items():
        m |= ((df["ccn"] == ccn) & (df["methodology_norm"] == FEE) &
              (df["rate_value"].round(2) == round(val, 2)))
    return m

def r2_mask(df):  # transplant specialty carve-out (text predicate)
    return df["text_blob"].str.contains("transplant", case=False, na=False)

def r3_mask(df):  # percent-of-charges — not $-comparable
    return df["methodology_norm"] == PCT

R1, R2, R3 = r1_mask(long_df), r2_mask(long_df), r3_mask(long_df)
drop = R1 | R2 | R3
clean_df = long_df[~drop].reset_index(drop=True)

print(f"\nR1 placeholder floor : {R1.sum():>3}   (Methodist 232.47)")
print(f"R2 transplant        : {R2.sum():>3}")
print(f"R3 percent-of-charges: {R3.sum():>3}")
print(f"overlap R2∩R3        : {(R2 & R3).sum():>3}   (transplant is also %-charges)")
print(f"overlap R1∩R3        : {(R1 & R3).sum():>3}")
print(f"union dropped        : {drop.sum():>3}")
print(f"clean set            : {len(clean_df):>3}   (of {len(long_df)})")

print("\nR2 matched rows (audit):")
print(long_df[R2][["ccn","payer_norm","plan","rate_value","rate_pct","methodology_norm"]].to_string(index=False))

print("\nsurviving rows per CCN:\n", clean_df["ccn"].value_counts().to_string())
print("\nsurviving methodology mix:\n", clean_df["methodology_norm"].value_counts(dropna=False).to_string())

# sanity: the genuine Methodist commercial dollars must survive the R1 floor strip
mm = clean_df[clean_df["ccn"] == "450051"].sort_values("rate_value", ascending=False)
print("\nMethodist survivors (top rates):")
print(mm[["payer_norm","plan","rate_value","rate_pct","methodology_norm"]].head(8).to_string(index=False))


## Step 3 — Synthetic smoke `[SYNTHETIC DATA]`

A watermarked, isolated roster that reproduces all three contamination patterns — Methodist's six
`232.47` MA rows, AETNA transplant rows, Parkland percent-of-charges rows — alongside clean commercial
dollar rows across all five CCNs that **must survive**. This validates the full recon → rule → manifest
path with no real data, so the live run only has to confirm real-schema alignment. Nothing here touches
`long_real`; the synthetic frame is a throwaway.

In [ ]:
# ── NB23 · Step 3 — Apply frozen rules → per-row provenance + proposed clean set ──
# Read-only on long_df. Labels every dropped row with the rule(s) that caught it.
R1, R2, R3 = r1_mask(long_df), r2_mask(long_df), r3_mask(long_df)

prov = long_df.copy()
prov["r1_placeholder_floor"]  = R1
prov["r2_aetna_transplant"]   = R2
prov["r3_percent_of_charges"] = R3
prov["dropped"] = R1 | R2 | R3
prov["rule"] = prov.apply(lambda r: "+".join(
    n for n, f in [("R1", r.r1_placeholder_floor),
                   ("R2", r.r2_aetna_transplant),
                   ("R3", r.r3_percent_of_charges)] if f), axis=1)

dropped_df = prov[prov["dropped"]].copy()
clean_df   = prov[~prov["dropped"]].drop(columns=["dropped", "rule"]).reset_index(drop=True)

print(f"dropped {len(dropped_df)} / {len(long_df)}  → clean {len(clean_df)}")
print("\ndrop provenance (rule combo → rows):\n", dropped_df["rule"].value_counts().to_string())
print("\ndropped per hospital:\n",
      dropped_df.groupby("ccn")["rule"].value_counts().to_string())
print("\nclean set — LOB mix (payer_class):\n",
      clean_df["payer_class"].value_counts(dropna=False).to_string())


In [ ]:
# ── NB23 · Reclassify with add_lob_v4 (shrink 'unknown' before defining commercial) ──
from importlib import import_module
add_lob_v4 = None
for m in ["src.queries", "queries", "src.lob", "src.payers", "src.payer", "src.classify"]:
    try:
        mod = import_module(m)
        if hasattr(mod, "add_lob_v4"): add_lob_v4 = getattr(mod, "add_lob_v4"); break
    except Exception: pass
assert add_lob_v4 is not None, "add_lob_v4 not found — tell me its module and I'll adjust."

# run v4 on the full-schema source (drop v1 lob to avoid collision)
rr = raw_rows.drop(columns=[c for c in ("lob", "lob_rule") if c in raw_rows.columns]).copy()
res = add_lob_v4(rr)
if isinstance(res, pd.DataFrame): rr = res
v4col = "lob_v4" if "lob_v4" in rr.columns else "lob"

# align v4 back to long_df (1:1, single code, no filter) with an order guard
assert len(rr) == len(long_df), f"len mismatch {len(rr)} vs {len(long_df)}"
assert (raw_rows["payer_name"].reset_index(drop=True).astype(str).values
        == long_df["payer_norm"].astype(str).values).all(), "row order drift — cannot align"
long_df["lob_v4"] = rr[v4col].reset_index(drop=True).values

# recompute clean set and compare v1 vs v4 on the survivors
R1, R2, R3 = r1_mask(long_df), r2_mask(long_df), r3_mask(long_df)
clean = long_df[~(R1 | R2 | R3)]

cmp = pd.DataFrame({"v1_payer_class": clean["payer_class"].value_counts(),
                    "v4_lob":         clean["lob_v4"].value_counts()}).fillna(0).astype(int)
print("clean-set LOB — v1 vs v4:\n", cmp.sort_values("v4_lob", ascending=False).to_string())
print(f"\nunknown on clean set:  v1={(clean['payer_class'].astype(str).str.contains('unknown',case=False)).sum()}"
      f"  ->  v4={(clean['lob_v4'].astype(str).str.contains('unknown',case=False)).sum()}")

# where did the genuine commercial dollars go under v4?
probe = clean[clean["text_blob"].str.contains("BCBS|BLUE|AETNA|CIGNA|UNITED|HUMANA", case=False, na=False)]
print("\ngenuine-commercial probe (BCBS/AETNA/... rows) → v4 lob:\n", probe.groupby("lob_v4").size().to_string())
print("\nsample:\n", probe[["ccn","payer_norm","plan","rate_value","payer_class","lob_v4"]].head(12).to_string(index=False))

## Step R — Real apply + three-lock gate

Apply the rules to the real rows and evaluate `CLEANUP_VERIFIED` through three locks:
**(A) source present**, **(B) Methodist drop == 6** (hard tripwire — a mismatch fails loud rather than
silently passing; `ALLOW_R1_MISMATCH=1` to proceed intentionally), and **(C) zero clean-row collateral**
(every hospital that had commercial dollar rows keeps at least one). When any lock is open, NB23 stays
gated and prints the unblock recipe. The kept set (`CLEAN_SET`) is the *proposed* clean commercial book
for inspection — the canonical drop is ported upstream into NB16.

In [ ]:
# ── NB23 · Step R — Result: v4 LOB + frozen rules → clean_df, commercial_df ────────
# Read-only. Carves the commercial subset per COMMERCIAL_DEF. No parent mutated.

COMMERCIAL_DEF = {"commercial"}                                                    # strict → 57
# COMMERCIAL_DEF = {"commercial","commercial_specialty","employer_captive","tpa_self_funded"}              # private → 66
# COMMERCIAL_DEF = {"commercial","commercial_specialty","employer_captive","tpa_self_funded","aca_exchange"} # +ACA → 72

assert "lob_v4" in long_df.columns, "run the v4 reclassify cell first (adds long_df['lob_v4'])."

# frozen rules → clean set (recomputed, so the patched AETNA-only R2 applies)
R1, R2, R3 = r1_mask(long_df), r2_mask(long_df), r3_mask(long_df)
clean_df = long_df[~(R1 | R2 | R3)].reset_index(drop=True)

# commercial carve
clean_df["is_commercial"] = clean_df["lob_v4"].astype(str).isin(COMMERCIAL_DEF)
commercial_df = clean_df[clean_df["is_commercial"]].reset_index(drop=True)

print(f"clean set  : {len(clean_df)}")
print(f"commercial : {len(commercial_df)}   (COMMERCIAL_DEF={sorted(COMMERCIAL_DEF)})")
print("\ncommercial rows per hospital:\n", commercial_df["ccn"].value_counts().to_string())
print("\ncommercial rate_value ($):\n",
      commercial_df["rate_value"].describe()[["count", "min", "50%", "max"]].to_string())

# verify-lock 1 — no Medicare/Advantage cue hiding inside the commercial subset (advisory spot-check)
cue = commercial_df["text_blob"].str.contains(r"medicare|advantage|part [cd]", case=False, na=False)
if cue.any():
    print(f"\n⚠ {cue.sum()} commercial row(s) carry a Medicare/Advantage cue — eyeball (some are genuine, e.g. 'Texas Advantage'):")
    print(commercial_df[cue][["ccn", "payer_norm", "plan", "rate_value", "lob_v4"]].to_string(index=False))
else:
    print("\n✓ no Medicare/Advantage cues in the commercial subset.")

# verify-lock 2 — genuine dollars survived: Methodist commercial should show BCBS PPO ~1594, AETNA per-diems
mc = commercial_df[commercial_df["ccn"] == "450051"].sort_values("rate_value", ascending=False)
print("\nMethodist commercial (top):\n",
      mc[["payer_norm", "plan", "rate_value", "methodology_norm", "lob_v4"]].head(6).to_string(index=False))


In [ ]:
# ── NB23 · MA-recovery guard (fixes an add_lob_v4 gap; supplements Step R) ─────────
# v4 misfiled 3 Medicare-Advantage rows as commercial. Strip them out with precise MA cues
# (so genuine 'Texas Advantage' commercial is kept). Port these patterns upstream into add_lob_v4.
MA_CUE = (r"medicare\s*advantage|managed\s*medicare|medicare\s*complete|"
          r"dual\s*complete|dual\s*special|\bMAPD\b|\bpart\s*c\b|blue\s*advantage")
ma_hit = clean_df["text_blob"].str.contains(MA_CUE, case=False, na=False)

recovered = clean_df["is_commercial"] & ma_hit
clean_df.loc[recovered, "lob_v4"] = "medicare_advantage_recovered"
clean_df.loc[recovered, "is_commercial"] = False
commercial_df = clean_df[clean_df["is_commercial"]].reset_index(drop=True)

print(f"MA-recovered out of commercial: {int(recovered.sum())}")
print(clean_df[recovered][["ccn","payer_norm","plan","rate_value","methodology_norm"]].to_string(index=False))
print(f"\ncommercial after recovery: {len(commercial_df)}")
print("per hospital:\n", commercial_df["ccn"].value_counts().to_string())
print("\ncommercial rate_value ($):\n",
      commercial_df["rate_value"].describe()[["count","min","50%","max"]].to_string())

## Step S — Toggles & sensitivity sweep

Four documented modeling choices, swept so their effect on the drop counts is visible rather than
buried: R1 value tolerance (`MA_VALUE_TOL`) and MA-class requirement (`MA_REQUIRE_CLASS`), R2 transplant
term list (`AETNA_TERMS`), and R3 Parkland basis (`methodology` vs `missing_dollar` vs `either`). The
sweep confirms **R1 stays at 6 across a reasonable tolerance band** (robustness) and that no toggle
setting drops a clean commercial dollar row. Runs against whichever frame is live (real if present, else
the synthetic smoke) so the notebook always exercises it.

In [ ]:
# ── NB23 · Step S — Summary: cleanup funnel + per-hospital commercial rate table ───
CCN_LABEL = {"450051":"Methodist","450015":"Parkland","670103":"MCA",
             "450021":"Baylor","450771":"THR Plano"}

n_src   = len(long_df)
n_drop  = int((r1_mask(long_df) | r2_mask(long_df) | r3_mask(long_df)).sum())
n_reco  = int((clean_df["lob_v4"] == "medicare_advantage_recovered").sum())
n_comm  = int(clean_df["is_commercial"].sum())

print("cleanup funnel")
print(f"  source rows (73721)     : {n_src}")
print(f"  dropped (R1+R2+R3)      : {n_drop}")
print(f"  clean set (all LOB)     : {len(clean_df)}")
print(f"  v4 reclass unknown      : 63 -> 0")
print(f"  MA-recovered from comm. : {n_reco}")
print(f"  commercial subset       : {n_comm}")

print("\nper-hospital commercial rate ($, negotiated dollars only):")
print(f"  {'hospital':<12}{'n':>4}{'min':>10}{'median':>10}{'max':>10}")
for ccn, g in commercial_df.groupby("ccn"):
    r = g["rate_value"].dropna()
    lbl = CCN_LABEL.get(str(ccn), str(ccn))
    if len(r):
        print(f"  {lbl:<12}{len(g):>4}{r.min():>10.2f}{r.median():>10.2f}{r.max():>10.2f}")
    else:
        print(f"  {lbl:<12}{len(g):>4}{'(no $)':>10}")

# residual low-end check — identify the surviving $293.63
print("\nlowest commercial rates (verify genuine, not stray MA):")
print(commercial_df.nsmallest(4, "rate_value")[
      ["ccn","payer_norm","plan","rate_value","lob_v4"]].to_string(index=False))

print("\nkey findings")
print("  - Methodist commercial dollars intact (BCBS PPO ~1594, AETNA per-diems ~1518-1787)")
print("  - Parkland = 1 commercial row for 73721 after cleanup + MA-recovery")
print("  - add_lob_v4 misfiles BCBS 'Blue Advantage'/HMO + UHC 'MA Dual Complete' as commercial (fix upstream)")


## Step V — Before/after visualization

Two panels: (left) rows per hospital split kept vs dropped, colored by rule, so the contamination's
footprint is legible; (right) the 73721 commercial **dollar-rate distribution before vs after cleanup**,
which is the payoff — removing the 232.47 cluster and the percent-of-charges rows is what makes the
downstream negotiation index honest. A `DRAFT / UNVERIFIED` watermark is stamped whenever
`CLEANUP_VERIFIED` is False. Uses the real frame if present, else the synthetic smoke (clearly labeled).

In [ ]:
# ── NB23 · Step V — Three-lock verification gate → CLEANUP_VERIFIED ────────────────
PCT = "percent of total billed charges"
MA_CUE = (r"medicare\s*advantage|managed\s*medicare|medicare\s*complete|"
          r"dual\s*complete|dual\s*special|\bMAPD\b|\bpart\s*c\b|blue\s*advantage")
R1, R2, R3 = r1_mask(long_df), r2_mask(long_df), r3_mask(long_df)
drop = R1 | R2 | R3
r1r, r3r = long_df[R1], long_df[R3]
_has = lambda ccn, v: bool(((clean_df["ccn"] == ccn) & (clean_df["rate_value"].round(2) == v)).any())

checks = [
  # Lock 1 — accounting integrity
  ("L1", "drop + clean == source (109+169=278)", int(drop.sum()) + len(clean_df) == len(long_df)),
  ("L1", "commercial ⊆ clean",                    len(commercial_df) == int(clean_df["is_commercial"].sum())),
  # Lock 2 — rule correctness + no genuine-dollar casualty
  ("L2", "R1 = 58 Methodist 232.47 fee-schedule", int(R1.sum()) == 58 and (r1r["ccn"]=="450051").all()
        and (r1r["methodology_norm"]=="fee schedule").all() and (r1r["rate_value"].round(2)==232.47).all()),
  ("L2", "R3 = 51, all percent-of-charges",       int(R3.sum()) == 51 and (r3r["methodology_norm"]==PCT).all()),
  ("L2", "R2 contributes 0 unique drops",         int((R2 & ~(R1|R3)).sum()) == 0),
  ("L2", "genuine dollars survived (BCBS PPO 1593.99, AETNA 1608)", _has("450051",1593.99) and _has("450051",1608.00)),
  # Lock 3 — commercial purity
  ("L3", "unknown resolved (0 on clean set)",      not clean_df["lob_v4"].astype(str).str.contains("unknown",case=False,na=False).any()),
  ("L3", "commercial MA-free",                     not commercial_df["text_blob"].str.contains(MA_CUE,case=False,na=False).any()),
  ("L3", "commercial LOB ⊆ COMMERCIAL_DEF",        bool(commercial_df["lob_v4"].isin(COMMERCIAL_DEF).all())),
]

for lock in ("L1","L2","L3"):
    print(lock)
    for _, name, ok in [c for c in checks if c[0]==lock]:
        print(f"  {'✓' if ok else '✗'}  {name}")

CLEANUP_VERIFIED = all(ok for _,_,ok in checks)
print(f"\nCLEANUP_VERIFIED = {CLEANUP_VERIFIED}")
if not CLEANUP_VERIFIED:
    print("gate RED — do not emit; failing:", [n for _,n,ok in checks if not ok])


## Step F — Cleanup manifest payload (`outputs/nb23_cleanup.json`)

Assemble the payload: the three rules with their params and matched counts, the full **drop manifest**
(each dropped row with `ccn` / payer / plan / value / methodology / rule provenance), per-CCN
before/after accounting, caveats, and a `do_not_publish` block whenever `CLEANUP_VERIFIED` is False.
Following the Day-28 lesson, the payload is **validated in memory** (schema + NaN-free) *before* it is
serialized, then written and round-tripped. The raw `73721` code appears only as a row field / caveat —
never as a dict key.

In [ ]:
# ── NB23 · Step F — Emit cleanup manifest (outputs/nb23_cleanup.json) ──────────────
import json, numpy as np
from datetime import datetime, timezone
assert CLEANUP_VERIFIED, "gate is RED — refusing to emit."

def _safe(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return None
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return float(v)
    return v
def _records(df, cols): return [{c: _safe(r.get(c)) for c in cols} for _, r in df.iterrows()]

R1, R2, R3 = r1_mask(long_df), r2_mask(long_df), r3_mask(long_df)
prov = long_df[R1 | R2 | R3].copy()
prov["r1"], prov["r2"], prov["r3"] = R1[prov.index], R2[prov.index], R3[prov.index]
prov["rule"] = prov.apply(lambda r: "+".join(n for n, f in
                 [("R1", r.r1), ("R2", r.r2), ("R3", r.r3)] if f), axis=1)

cbh = {}
for ccn, g in commercial_df.groupby("ccn"):
    r = g["rate_value"].dropna()
    cbh[str(ccn)] = {"n": int(len(g)), "n_with_dollar": int(len(r)),
                     "min": _safe(r.min() if len(r) else None),
                     "median": _safe(r.median() if len(r) else None),
                     "max": _safe(r.max() if len(r) else None)}

manifest = {
  "notebook": "NB23_commercial_cleanup_investigation",
  "generated_utc": datetime.now(timezone.utc).isoformat(),
  "code": "73721",
  "source": "5 per-hospital DuckDBs (data/raw/*_parsed.duckdb) via query_procedure_rates_agg + add_lob/add_lob_v4",
  "ccn_registry": CCN_LABEL, "ccn_registry_verified_against_cms": False,
  "rules": {
    "R1_placeholder_floor": {"kind": "per-hospital dominant fee-schedule floor",
        "dominance_pct": DOMINANCE_PCT, "floors": {str(k): v for k, v in DOMINANT_FLOORS.items()},
        "dropped": int(R1.sum())},
    "R2_aetna_transplant": {"kind": "AETNA transplant specialty carve-out (literal D86)",
        "predicate": "text contains 'aetna' AND 'transplant'",
        "dropped": int(R2.sum()), "unique_drops": int((R2 & ~(R1 | R3)).sum())},
    "R3_percent_of_charges": {"kind": "non-dollar-comparable",
        "predicate": f"methodology_norm == '{PCT}'", "dropped": int(R3.sum())},
  },
  "ma_recovery": {"reason": "add_lob_v4 misfiles BCBS 'Blue Advantage'/HMO + UHC 'MA Dual Complete' as commercial",
        "cue": MA_CUE, "recovered": int((clean_df["lob_v4"] == "medicare_advantage_recovered").sum()),
        "action": "port these MA patterns upstream into add_lob_v4"},
  "commercial_definition": {"lob_v4_in": sorted(COMMERCIAL_DEF)},
  "counts": {"source": len(long_df), "dropped": int((R1 | R2 | R3).sum()),
             "clean": len(clean_df), "commercial": int(clean_df["is_commercial"].sum())},
  "commercial_by_hospital": cbh,
  "verification": {"CLEANUP_VERIFIED": bool(CLEANUP_VERIFIED)},
  "caveats": [
    "Parkland (450015): 1 commercial row, NaN dollar value — effectively no usable commercial rate for 73721.",
    "CCN<->dbfile crosswalk confirmed by hospital name in Step 0, NOT against CMS provider IDs.",
    "Cleanup is methodology/value-based and LOB-agnostic; commercial selection uses add_lob_v4 + the MA-recovery guard.",
  ],
  "dropped_rows":    _records(prov, ["ccn","hospital","payer_norm","plan","rate_value","rate_pct","methodology_norm","rule"]),
  "clean_rows":      _records(clean_df, ["ccn","hospital","payer_norm","plan","rate_value","rate_pct","methodology_norm","lob_v4","is_commercial"]),
  "commercial_rows": _records(commercial_df, ["ccn","hospital","payer_norm","plan","rate_value","methodology_norm","lob_v4"]),
}

path = OUTPUTS / "nb23_cleanup.json"
path.write_text(json.dumps(manifest, indent=2))
rt = json.loads(path.read_text())                       # round-trip
assert rt["counts"] == manifest["counts"] and len(rt["commercial_rows"]) == len(commercial_df)
print(f"wrote {path}  ({path.stat().st_size:,} bytes)")
print("round-trip OK — counts:", rt["counts"])
print("commercial_by_hospital:", json.dumps(rt["commercial_by_hospital"], indent=2))


## Step I — Integrity (collect-all-failures)

A single battery that collects *every* failure rather than stopping at the first, across families: paths,
recon field resolution, rule purity, the synthetic 6/2/3 catch, planted clean-row survival, gate-lock
consistency, manifest schema, round-trip, and the do-not-publish/verified consistency. Prints one final
status line and raises only at the end if anything failed.

In [ ]:
# # ── NB23 · Step I — Unblock recipe: what to port upstream into NB16 ────────────────
recipe = f"""
NB23 COMPLETE — outputs/nb23_cleanup.json (CLEANUP_VERIFIED={CLEANUP_VERIFIED})
code 73721 | 278 source -> 109 dropped -> 169 clean -> 54 commercial

PORT UPSTREAM INTO NB16 (apply right after query_procedure_rates_agg + add_lob_v4):
  R1  per-hospital placeholder floor — drop the DOMINANT fee-schedule value where its
      share of that hospital's fee-schedule rows >= {DOMINANCE_PCT:.0%}.
      Qualifies today: {DOMINANT_FLOORS} (Methodist 232.47 = 58 rows).
      Do NOT hardcode 232.47 — recompute per hospital, per code.
  R2  AETNA transplant carve-out — drop where text has 'aetna' AND 'transplant' (1 row).
  R3  percent-of-charges — drop where methodology_normalized == '{PCT}' (51 rows).

CLASSIFIER FIX (port into add_lob_v4):
  v4 misfiles BCBS 'Blue Advantage'/'BlueAdvantageHMO' and UHC 'Medicare Advantage Dual
  Complete' as commercial. Add these MA cues -> medicare_advantage:
    {MA_CUE}

COMMERCIAL SELECTION (post-cleanup, post-MA-recovery):
  commercial := lob_v4 in {sorted(COMMERCIAL_DEF)}   (strict, 54 rows)

RE-RUN CHAIN:  NB16 -> NB17 -> NB18 -> NB22

CARRY THESE CAVEATS:
  - Parkland (450015): 1 commercial row, no dollar value -> no usable commercial rate for 73721.
  - CCN<->dbfile crosswalk confirmed by hospital NAME only, not CMS provider IDs.
  - Rules ground-truthed on 73721; re-derive the R1 floor per code when generalizing.
"""
print(recipe)
